In [1]:
import importlib.util, subprocess, sys

_pkgs = ["langgraph", "langchain_community", "langchain_openai", "langsmith", "langgraph-supervisor"]
_missing = [p for p in _pkgs if importlib.util.find_spec(p.replace("-", "_").split("[")[0]) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _pkgs)

In [2]:
# Environment Variable Initialization

import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from the notebook's working directory

def _set_if_undefined(var_name: str):
    value = os.environ.get(var_name, "").strip()
    if value:
        masked = value[:6] + "*" * (min(20, len(value) - 6))
        print(f"  ✅ {var_name}: {masked}")
    else:
        print(f"  ❌ {var_name}: not set")

# ---- Environment Variables Required ----

print("Checking environment variables...")
_set_if_undefined("OPENAI_API_KEY")
_set_if_undefined("LANGSMITH_TRACING")
_set_if_undefined("LANGSMITH_API_KEY")  # https://docs.langchain.com/langsmith/observability
_set_if_undefined("MODEL")
_set_if_undefined("OPENWEATHER_API_KEY")  # https://openweathermap.org/api
print("Done.")

Checking environment variables...
  ✅ OPENAI_API_KEY: sk-pro********************
  ✅ LANGSMITH_TRACING: true
  ✅ LANGSMITH_API_KEY: lsv2_p********************
  ✅ MODEL: gpt-4o*****
  ✅ OPENWEATHER_API_KEY: 8dd1b8********************
Done.


In [3]:
# Multi-Agent Orchestration with LangGraph:
#- Supervisor agent coordinates between specialized workers.
#- Workers: weather reporting agent, dressing planner agent.

# ---- Imports ----

import os, warnings
import requests
from langchain_openai import ChatOpenAI
from typing import Annotated
from langgraph_supervisor import create_supervisor
from langgraph.prebuilt import create_react_agent
from langgraph.warnings import LangGraphDeprecatedSinceV10
from langchain_core.tools import tool

# Silence LangGraph v1.0 deprecation warnings emitted by create_supervisor /
# create_react_agent so the demo output stays clean.
warnings.filterwarnings("ignore", category=LangGraphDeprecatedSinceV10)

# ---- LLM Setup ----

# Load the default model from environment variables
default_model = os.environ["MODEL"]

# Initialize the LLM (Large Language Model) interface
llm = ChatOpenAI(model=default_model)

# ---- Node Definitions ----
@tool
def weather_reporting_tool(city: Annotated[str, "name of the city"]):
    """Tool to fetch current weather data for a given city from OpenWeatherMap."""
    resp = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={
            "q": city,
            "appid": os.environ["OPENWEATHER_API_KEY"],
            "units": "metric",  # temperatures in Celsius
        },
        timeout=10,
    )
    resp.raise_for_status()
    data = resp.json()
    return {
        "weather": {
            "name": data["name"],
            "main": data["weather"][0]["main"],
            "description": data["weather"][0]["description"],
            "units": "metric (Celsius)",
            **data["main"],   # temp, feels_like, temp_min, temp_max, pressure, humidity
            "wind_speed": data.get("wind", {}).get("speed"),
        }
    }


# Weather Reporting Agent
weather_reporting_agent = create_react_agent(
    llm,
    tools=[weather_reporting_tool],
    name="weather_reporting_agent",
    prompt=(
        "You are a weather reporter. Report current weather for the provided city. "
        "You may use tools. Temperatures are in Celsius. Do not suggest what to wear."
    )
)

# Dressing Planner Agent
dressing_planner_agent = create_react_agent(
    llm,
    tools=[],
    name="dressing_planner_agent",
    prompt=(
        "You suggest dressing options based on the current weather. "
        "Prioritize 'feels like' temperature and consider wind conditions."
    )
)

# ---- Supervisor Setup ----

# System prompt guiding the supervisor's behavior
system_prompt = (
    "# Role and Objective"
    "You are a Supervisor Agent tasked with managing a conversation between two specialized workers: "
    "`weather_reporting_agent` and `dressing_planner_agent`."
    "Your goal is to orchestrate their actions to resolve the user's request efficiently."
    "# Instructions"
    " - Persist through multiple steps until the task is fully complete."
    " - Always select the next worker based on context."
    " - Think step-by-step before choosing a worker and after receiving results."
    "# Reasoning Steps"
    "1. Analyze current state."
    "2. Plan the next best action."
    "3. Reflect after worker output."
    "4. Repeat until completion."
    "# Tool/Worker Use"
    "- `weather_reporting_agent`: gather or analyze weather."
    "- `dressing_planner_agent`: suggest clothing based on weather."
)

# Create supervisor 
builder = create_supervisor(
    agents=[weather_reporting_agent, dressing_planner_agent],
    model=llm,
    output_mode="last_message",
    prompt=(system_prompt)
)


  

In [4]:
# Compile the graph
graph = builder.compile()


In [5]:
# ---- Simulation ----
# Stream updates and print only the latest message from each agent step,
# instead of the verbose debug=True trace.

for chunk in graph.stream({"messages": [("user", "Stockholm")]}):
    for node, update in chunk.items():
        msgs = update.get("messages") if isinstance(update, dict) else None
        if msgs:
            last = msgs[-1]
            content = getattr(last, "content", "")
            if content:
                print(f"[{node}] {content}")
                print("=" * 60)


[supervisor] I need more information about your request regarding Stockholm. Are you looking for the weather report or clothing suggestions for a certain activity or weather condition?
